# 05 · Uçtan Uca Sistem: Belge → Tür → İçerik

```
Belge (resim / PDF) → OCR → Metin temizleme → Tür tahmini → Türe özel içerik çıkarımı → "Bu bir CV: ..."
```

| Hocanın maddesi | Bölüm |
|---|---|
| Final 5 · Yeni belge → OCR → tür tahmini | 4–6 |
| İlk istek · Belgenin içeriğini sunmak (ör. CV ise ad, üniversite, bölüm) | 3–6 |
| 8 Ekim görüşmesi · Belgenin ne hakkında olduğunu anlatan kısa özet (anahtar kelimeler + 2 cümle) | 4–6 |
| Asistan önerisi · Tablo tespiti | 4–6 |
| İçerik çıkarımının değerlendirilmesi | 7 |
| Demo arayüzü | 8 |

**Girdi:** `models/best_model.joblib` (04) · Önce 04 çalışmış olmalı. **10_son_model** çalıştırıldıysa onun modeli (metin + tablo özelliği, "emin değilim" eşiği) otomatik kullanılır. GPU gerekmez.

## 1. Kurulum

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract opencv-python-headless pymupdf spacy gradio symspellpy
!python -m spacy download en_core_web_sm -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys, json, importlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, joblib
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
MODELS = ROOT / "models"
OWN = ROOT / "kendi_belgelerim"; OWN.mkdir(exist_ok=True)   # kendi belgelerinizi buraya koyun
# Sınıf listesi tek yerden: Drive'daki siniflar.json (06'nın son hücresi yazar). Dosya yoksa ilk 4 sınıf kullanılır.
import json
CFG_FILE = ROOT / "siniflar.json"
CLASSES = json.loads(CFG_FILE.read_text())["CLASSES"] if CFG_FILE.exists() else ["email", "form", "invoice", "resume"]
print("Sınıflar:", CLASSES, "(siniflar.json)" if CFG_FILE.exists() else "(varsayılan)")
SEED = 42
sys.path.insert(0, str(ROOT))

## 2. İçerik çıkarım modülü

Belge türü belirlendikten sonra, o türe özel alanlar iki yöntemle çıkarılır:

| Yöntem | Kullanıldığı yer |
|---|---|
| **Kurallar (regex)** | E-posta adresi, telefon, tarih, tutar, fatura no, `From:` / `Subject:` başlıkları, `Alan: değer` çiftleri |
| **İsim tanıma (spaCy NER)** | Kişi adı (CV), şirket adı (fatura) |
| **Bölüm başlıkları** | CV'de `EDUCATION` altındaki satırlardan üniversite ve bölüm, `SKILLS` altından beceriler |

| Tür | Çıkarılan alanlar |
|---|---|
| CV | ad-soyad, üniversite, bölüm/derece, e-posta, telefon, beceriler |
| Fatura | fatura no, tarih, firma, toplam tutar |
| E-posta | gönderen, alıcı, konu, tarih, gövde özeti |
| Form | başlık, alan–değer çiftleri, tarih |

**Konu özeti (her tür için):** TF-IDF ağırlıklarına göre belgenin en ayırt edici kelimeleri seçilir ve bu kelimeleri en çok içeren
2 cümle özet olarak gösterilir (çıkarımsal özetleme). Örnek: *"Bu bir e-posta. Konusu / anahtar kelimeler: smoking, cancer, study."*

**Tablo tespiti:** Görüntüdeki yatay ve dikey çizgiler morfolojik işlemlerle bulunur; OCR'ın bozduğu tablo bilgisi böylece ayrıca raporlanır.

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/extract.py
"""Belge türüne özel içerik çıkarımı (kural tabanlı + spaCy NER).

Girdi: OCR'dan gelen temizlenmiş metin (text_clean.clean_text çıktısı) ve tahmin edilen tür.
Çıktı: {"alanlar": {...}, "ozet": "Bu bir CV: ..."}
"""
import re

try:
    import spacy
    _NLP = spacy.load("en_core_web_sm")
except Exception:  # spaCy yoksa sadece kurallar çalışır
    _NLP = None

# ---------------------------------------------------------------- ortak kalıplar
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
PHONE = re.compile(r"(?:\+?\d{1,3}[\s.-]?)?(?:\(\d{2,4}\)|\d{2,4})[\s.-]?\d{3}[\s.-]?\d{2,4}(?:[\s.-]?\d{2})?")
# v2 (21 Ekim, elle değerlendirme): belge numarası (50145702) ve yayın sayfa aralığı (30502-30509) telefon sanılıyordu.
# Artık ya "tel/phone" etiketi ya da telefon biçimi (alan kodu + 3 + 4 hane, ayraçlı) gerekiyor.
PHONE_LABELED = re.compile(r"\b(?:tel|telephone|phone|mobile|cell|gsm)\b\.?\s*(?:no\.?|number)?\s*[:#.]?\s*(\+?[\d(][\d\s().-]{6,18}\d)", re.I)
PHONE_FORMATTED = re.compile(r"(?<![\d-])(?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)\s?|\d{3}[\s.-])\d{3}[\s.-]\d{4}(?![\d-])")


def find_phone(text):
    m = PHONE_LABELED.search(text)
    if m and sum(c.isdigit() for c in m.group(1)) >= 7:
        return " ".join(m.group(1).split())
    m = PHONE_FORMATTED.search(text)
    return m.group(0).strip() if m else None
URL = re.compile(r"\b(?:https?://|www\.)\S+|\b(?:github|linkedin)\.com/\S+", re.I)
MONTHS = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?"
DATE = re.compile(
    rf"\b(?:\d{{1,2}}[/.-]\d{{1,2}}[/.-]\d{{2,4}}|{MONTHS}\s+\d{{1,2}},?\s+(?:19|20)\d{{2}}|"
    rf"\d{{1,2}}[\s.-]+{MONTHS}[\s.,-]*(?:19|20)?\d{{2}}|\d{{4}}-\d{{2}}-\d{{2}})\b",   # v2: "10.JULY.2014"
    re.I)


def find_date(text):
    """İlk geçerli tarih. v2: "70.00.00" gibi gün/ay aralığı dışındaki sayı grupları tarih sayılmaz."""
    for m in DATE.finditer(text or ""):
        d = m.group(0).strip()
        parts = re.split(r"[/.-]", d)
        if all(p.isdigit() for p in parts) and len(parts) == 3 and len(parts[0]) <= 2:
            a, b = int(parts[0]), int(parts[1])
            if not ((1 <= a <= 12 and 1 <= b <= 31) or (1 <= a <= 31 and 1 <= b <= 12)):
                continue
        return d
    return None


# v2: Avrupa yazımı (100.000,00) da tutar; "100.000,00" artık "100.00" diye kesilmiyor
_NUM = r"(?:\d{1,3}(?:\.\d{3})+(?:,\d{2})?|\d{1,3}(?:,\d{3})+(?:\.\d{2})?|\d+(?:[.,]\d{2})?)"
_NUM_DEC = r"(?:\d{1,3}(?:[.,]\d{3})+[.,]\d{2}|\d+[.,]\d{2})"   # OCR ayraçları karıştırabiliyor: 20.000.00
_CUR = r"(?:usd|eur|gbp|tl|try)\b"
MONEY = re.compile(rf"(?<![\d.,])(?:[$€£₺]\s?{_NUM}(?:\s?{_CUR})?|{_NUM_DEC}(?:\s?{_CUR})?|{_NUM}\s?{_CUR})(?![\d]|[.,]\d)", re.I)

DEGREE_WORDS = ("bachelor", "master", "phd", "ph.d", "b.s", "b.sc", "m.s", "m.sc", "b.a", "m.a", "mba",
                "degree", "engineering", "science", "lisans", "mühendisliği", "muhendisligi")
SCHOOL_WORDS = ("university", "universit", "college", "institute", "school", "academy", "üniversite", "universite")
SECTION_HEADS = ("education", "experience", "work experience", "employment", "skills", "technical skills",
                 "projects", "references", "publications", "certifications", "languages", "interests",
                 "objective", "summary", "awards", "activities")


def _lines(text):
    return [l.strip() for l in text.splitlines() if l.strip()]


def _ents(text, labels):
    if _NLP is None or not text:
        return []
    doc = _NLP(text[:20000])
    out = []
    for e in doc.ents:
        t = " ".join(e.text.split())
        if e.label_ in labels and len(t) > 1 and t not in out:
            out.append(t)
    return out


def _first(pattern, text, group=0):
    m = pattern.search(text)
    return m.group(group).strip() if m else None


def _section(lines, names):
    """Başlığı `names` içinde olan bölümün satırlarını döndürür (bir sonraki başlığa kadar)."""
    out, inside = [], False
    for l in lines:
        key = re.sub(r"[^a-z ]", "", l.lower()).strip()
        is_head = key in SECTION_HEADS and len(l) < 40
        # v2: form tipi CV'lerde başlık açıklamayla birlikte gelir: "EDUCATION/TRAINING (Begin with ...)"
        m = _LONG_HEAD.match(l)
        if not is_head and m:
            is_head, key = True, m.group(1).lower()
        if is_head:
            if inside:
                break
            inside = key in names
            continue
        if inside:
            out.append(l)
    return out


_LONG_HEAD = re.compile(r"^\s*(EDUCATION|EXPERIENCE|EMPLOYMENT|PUBLICATIONS|SKILLS)\b(?:\s*/\s*[A-Z]+)?\s*[(:]")

_NOT_NAME = SECTION_HEADS + SCHOOL_WORDS + DEGREE_WORDS + ("resume", "curriculum", "vitae", "cv", "page",
                                                           "street", "avenue", "road", "phone", "email", "address",
                                                           # v2: CV başlıkları ve form alanları ad sanılıyordu
                                                           "biographical", "sketch", "personal", "data", "principal",
                                                           "investigator", "program", "director", "grant", "number",
                                                           "position", "title", "research", "associate", "professor",
                                                           "senior", "department", "chief", "staff", "form", "birth")
def _is_name_token(t):
    """"Sena" · "ÖZKAN" · "M." · "Jean-Paul" (Türkçe harfler dahil)."""
    if len(t) == 2 and t[0].isalpha() and t[0].isupper() and t[1] == ".":
        return True
    core = t.replace("-", "")
    return len(t) >= 2 and core.isalpha() and t[0].isupper() and (t[1:].replace("-", "").islower() or t.isupper())
_NAME_SUFFIX = re.compile(r"^(?:M\.?D\.?|Ph\.?D\.?|Jr\.?|Sr\.?)$")
_NAME_LABEL = re.compile(r"^\s*(?:full\s+)?name\b\s*[:;]?\s*(.*)$", re.I)
_NOT_NAME_EXTRA = ("institution", "location", "field", "study", "year", "conferred", "birthdate", "assistant",
                   "fellow", "scientist", "lecturer", "instructor", "manager", "engineer", "head", "member")


def _name_candidate(segment):
    """Satır parçasının başındaki ad-soyadı döndürür: büyük harfle başlayan 2-4 kelime (baş harf "M." olabilir).
    Başlık/kurum kelimesine gelince durur: "Cecilia M. Giachelli Research Associate" → "Cecilia M. Giachelli"."""
    toks, out = segment.strip().split(), []
    for t in toks[:6]:
        core = t.rstrip(",")
        if not _is_name_token(core) or core.lower().strip(".") in _NOT_NAME + _NOT_NAME_EXTRA:
            break
        out.append(core)
        if t.endswith(",") or len(out) == 4:
            break
    if sum(len(w.strip(".")) >= 2 for w in out) < 2:
        return None
    name = " ".join(out)
    rest = toks[len(out):]
    if rest and _NAME_SUFFIX.match(rest[0].strip(",")):     # "Glenn R. Cunningham, M.D."
        name += ", " + rest[0].strip(",")
    return name


def _resume_name(lines):
    # 1) "Name" / "Full Name" etiketi: aynı satırda ya da sonraki 3 satırda
    for i, l in enumerate(lines[:20]):
        m = _NAME_LABEL.match(l)
        if not m:
            continue
        for cand in [m.group(1)] + lines[i + 1:i + 4]:
            for seg in re.split(r"\s*\|\s*|\s{2,}", cand or ""):
                n = _name_candidate(seg)
                if n:
                    return n
    # 2) üst satırlarda satır başında ad (form tipi CV: "Glenn R. Cunningham, M.D. VA Medical Center ...")
    for l in lines[:12]:
        n = _name_candidate(re.split(r"\s*\|\s*", l)[0])
        if n:
            return n
    # 3) eski yöntem: spaCy kişi adı ya da ad gibi görünen satır
    persons = [p for p in _ents("\n".join(lines[:8]), {"PERSON"}) if _looks_like_name(p)]
    return persons[0] if persons else next((l for l in lines[:6] if _looks_like_name(l)), None)


# açık derece: noktalı kısaltmalar (B.A., Ph.D., BS.) ya da "BA in ..."; "MA. 02138" gibi eyalet kodları sayılmaz
_STRONG_DEGREE = re.compile(r"\b(?:B\.\s?A\.?|B\.?\s?S\.|B\.\s?Sc\.?|M\.\s?S\.?|M\.\s?Sc\.?|M\.\s?A\.|Ph\.?\s?D\.?|M\.\s?D\.|"
                            r"BSc|MSc|MBA|(?:BA|BS|MS|MA)(?=\s+in\s))|\b(?:bachelor|master|doctorate)\b")
_YEAR = re.compile(r"\b(?:19|20)\d{2}\b")


def _looks_like_name(line):
    """2-4 kelimelik, harflerden oluşan, büyük harfle başlayan, başlık/okul/adres olmayan satır."""
    if any(ch.isdigit() for ch in line) or any(ch in line for ch in "@:/|()"):
        return False
    words = line.replace(",", " ").replace(".", " ").split()
    if not 2 <= len(words) <= 4 or not all(w.isalpha() for w in words) or sum(len(w) >= 2 for w in words) < 2:
        return False
    low = line.lower()
    if any(k in low.split() or k in low for k in _NOT_NAME if len(k) > 3) or low in _NOT_NAME:
        return False
    return all(w[0].isupper() for w in words)


# ---------------------------------------------------------------- türe özel çıkarıcılar
def extract_resume(text):
    lines = _lines(text)
    name = _resume_name(lines)

    edu = _section(lines, {"education"}) or lines
    schools = [l for l in edu if any(w in l.lower() for w in SCHOOL_WORDS)]
    # v2: eğitim tablosu satırını (yıl ya da derece içeren okul satırı) tercih et; unvan satırındaki kurumu değil
    school = next((l for l in schools if _YEAR.search(l) or _STRONG_DEGREE.search(l)), schools[0] if schools else None)
    if school is None:
        orgs = [o for o in _ents("\n".join(edu), {"ORG"}) if any(w in o.lower() for w in SCHOOL_WORDS)]
        school = orgs[0] if orgs else None
    # v2: önce açık derece kısaltması (B.A., Ph.D. ...) olan satır; sonra genel kelimeler (uzun açıklama ve tablo başlığı hariç)
    name_key = name.split(",")[0] if name else None
    degree = next((l for l in edu if _STRONG_DEGREE.search(l) and l != school
                   and not (name_key and name_key in l)), None)   # ad satırındaki "M.D." derece satırı değil
    if degree is None and school and _STRONG_DEGREE.search(school):   # "BA in Biochemistry (1978, University of Dublin)"
        degree = school
    if degree is None:
        degree = next((l for l in edu if any(w in l.lower() for w in DEGREE_WORDS) and l != school
                       and len(l.split()) <= 15 and not re.search(r"\binstitution\b|\byear\b", l, re.I)), None)
    if school:
        school = re.sub(r"^\s*education\s*[:;-]\s*", "", school, flags=re.I) or None

    skills_lines = _section(lines, {"skills", "technical skills"})
    skills = None
    if skills_lines:
        items = re.split(r"[,;•·|]|\s-\s", " ".join(skills_lines[:4]))
        skills = [s.strip(" :.") for s in items if 1 < len(s.strip()) < 30][:10] or None

    return {
        "ad_soyad": name,
        "universite": school,
        "bolum_derece": degree,
        "e_posta": _first(EMAIL, text),
        "telefon": find_phone(text),
        "beceriler": skills,
    }


_NOT_COMPANY = ("invoice", "quantity", "description", "price", "amount", "total", "date", "bill to",
                "ship to", "page", "tax", "qty", "item")
_CORP = re.compile(r"\b(?:inc|corp|corporation|co|company|ltd|limited|llc|plc|group|gmbh|a\.?ş|ltd\.?\s*şti)\b\.?", re.I)


def _invoice_company(lines):
    top = [l for l in lines[:12] if not any(w in l.lower() for w in _NOT_COMPANY) and re.search(r"[A-Za-z]{3}", l)]
    corp = next((l for l in top if _CORP.search(l)), None)
    if corp:
        return corp
    orgs = [o for o in _ents("\n".join(top), {"ORG"}) if not any(w in o.lower() for w in _NOT_COMPANY)]
    if orgs:
        return orgs[0]
    # antetli kâğıt: ilk 3 satırdan büyük harf ağırlıklı olan
    def upper_ratio(l):
        letters = [c for c in l if c.isalpha()]
        return sum(c.isupper() for c in letters) / len(letters) if letters else 0
    return next((l for l in top[:3] if upper_ratio(l) > 0.6 and len(l) > 4), None)


def _invoice_amounts(lines):
    """v2: tablo satırlarında miktar ve birim fiyat da para gibi görünür (10.00, 50.00). Her satırın en sağdaki
    tutarı (TUTAR sütunu) alınır; belgede en az iki "$"lı tutar varsa sadece onlar tutulur."""
    vals = []
    for l in lines:
        found = [m.strip().rstrip(",.") for m in MONEY.findall(l)]
        if found:
            vals.append(found[-1])
    dollars = [v for v in vals if v[:1] in "$€£₺"]
    vals = dollars if len(dollars) >= 2 else vals
    out = list(dict.fromkeys(vals))
    return out[:5] or None


def extract_invoice(text):
    text = re.sub(r"(\d)[,.]\s+(\d{3})\b", r"\1,\2", text)   # OCR: '1, 250.00' -> '1,250.00'
    lines = _lines(text)
    inv_no = _first(re.compile(r"invoice\s*(?:no\.?|n0\.?|number|num|#)\s*[:.]?\s*((?=[A-Z0-9/-]*\d)[A-Z0-9][A-Z0-9/-]{1,20})", re.I), text, 1)
    # v2: önce kesin toplam ifadeleri (net amount, total due ...), sonra genel "total", en son "amount"
    total = None
    for pat in (r"(?:net\s+amount|total\s+amount\s+due|amount\s+due|balance\s+due|grand\s+total|total\s+due|invoice\s+total|amount\s+payable)",
                r"\btotal", r"\bamount"):
        found = []
        for m in re.finditer(pat + r"\b[^\n$€£\d]{0,20}", text, re.I):
            mm = MONEY.match(text, m.end()) or MONEY.search(text[m.end():m.end() + 25])
            if mm:
                found.append(mm.group(0).strip().rstrip(",."))
        if found:
            total = found[-1]
            break
    company = _invoice_company(lines)
    return {
        "fatura_no": inv_no,
        "tarih": _header(text, "date|invoice\\s+date|dated") or find_date(text),
        "firma": company,
        "toplam_tutar": total,
        "tutarlar": _invoice_amounts(lines),
    }


_LABEL_ONLY = re.compile(r"^\s*(?:from|to|cc|ce|co|bcc|subject|subj|re|date|sent|sender|recipient)\s*[:;]\s*$", re.I)


def _header(text, names):
    # v2: değer aynı satırda olmalı ([ \t]*); eskiden \s* satır sonunu da atladığı için "From:" satırının
    # değeri bir alt satırdaki "Date:" sanılıyordu. Değer başka bir etiketse kabul edilmez.
    pat = re.compile(rf"^[ \t]*(?:{names})[ \t]*[:;][ \t]*(\S[^\n]*)$", re.I | re.M)
    for m in pat.finditer(text):
        val = m.group(1).strip()
        if not _LABEL_ONLY.match(val):
            return val
    return None


# OCR "Cc:" etiketini sık sık "Ce:" ya da "co:" diye okuyor
_EMAIL_LABELS = {"from": "from", "sender": "from", "to": "to", "recipient": "to", "cc": "cc", "ce": "cc", "co": "cc",
                 "bcc": "cc", "subject": "subject", "subj": "subject", "re": "subject", "date": "date", "sent": "date"}
_HEAD_PAT = re.compile(r"^\s*(?:from|to|cc|ce|co|bcc|subject|subj|re|date|sent|sender|recipient)\s*[:;]", re.I)
_WEEKDAY_TIME = re.compile(r"\b(?:mon|tue|wed|thu|fri|sat|sun)[a-z]*\b.*\d{1,2}:\d{2}|"
                           r"\b\d{1,2}:\d{2}\s*(?:am|pm)\b", re.I)


def _label_block(lines):
    """OCR etiket sütununu (From/Date/Subject/To/Cc) değerlerden ayrı bir blok olarak okuduğunda
    etiketleri sırayla değerlerle eşleştirir. Hizalama tarih satırına göre yapılır."""
    for i in range(len(lines)):
        run = []
        j = i
        while j < len(lines) and _LABEL_ONLY.match(lines[j]):
            run.append(_EMAIL_LABELS[re.match(r"\s*([a-z]+)", lines[j], re.I).group(1).lower()])
            j += 1
        if len(run) < 2:
            continue
        rest = lines[j:j + len(run) + 6]
        if "date" in run:
            d = next((k for k, l in enumerate(rest) if DATE.search(l) or _WEEKDAY_TIME.search(l)), None)
            start = d - run.index("date") if d is not None else None
        else:
            start = None
        if start is None or start < 0:
            start = next((k for k, l in enumerate(rest) if not re.fullmatch(r"[\d\s.-]+", l)), 0)
        vals = rest[start:start + len(run)]
        if len(vals) < len(run):
            return {}, None
        out = {}
        for lab, v in zip(run, vals):
            if lab == "subject":
                v = re.sub(r"^\s*re\s*[:;]\s*", "RE: ", v, flags=re.I)
            out.setdefault(lab, v.strip())
        return out, j + start + len(run)        # gövdenin başladığı satır
    return {}, None


def extract_email(text):
    lines = _lines(text)
    # 1) etiket bloğu + değer bloğu (iki sütunlu başlık)
    block, body_start = _label_block(lines)
    sender, to = block.get("from"), block.get("to")
    subject, date = block.get("subject"), block.get("date")
    # 2) etiket ve değer aynı satırda
    sender = sender or _header(text, "from|sender")
    to = to or _header(text, "to|recipient")
    subject = subject or _header(text, "subject|subj|re")
    date = date or _header(text, "date|sent")
    # 3) etiketler hiç okunmamış (Outlook çıktısı): gönderen, gün+saat satırı, alıcılar, "Subject:"
    if not sender:
        subj_i = next((k for k, l in enumerate(lines[:15]) if re.match(r"\s*(?:subject|subj)\s*[:;]", l, re.I)), None)
        d_i = next((k for k, l in enumerate(lines[:6]) if _WEEKDAY_TIME.search(l) and DATE.search(l)), None)
        if d_i is not None and d_i >= 1 and (subj_i is None or d_i < subj_i):
            sender = lines[d_i - 1].strip(" '‘’")
            date = date or lines[d_i]
            if not to and subj_i is not None and subj_i > d_i + 1:
                to = " ".join(l.strip(" '‘’") for l in lines[d_i + 1:subj_i])[:200]
            body_start = max(body_start or 0, (subj_i + 1) if subj_i is not None else d_i + 1)
    date = date or find_date(text)
    # gövde: başlık satırlarından sonra
    if body_start is None:
        head_idx = [k for k, l in enumerate(lines[:20]) if _HEAD_PAT.match(l)]
        body_start = head_idx[-1] + 1 if head_idx else 0
    body = [l for l in lines[body_start:] if not _HEAD_PAT.match(l)]
    return {
        "gonderen": sender,
        "alici": to,
        "konu": subject,
        "tarih": date,
        "e_postalar": EMAIL.findall(text)[:5] or None,
        "govde_ozeti": " ".join(body[:3])[:200] or None,
    }


def extract_form(text):
    lines = _lines(text)
    title = next((l for l in lines[:8] if l.isupper() and len(l.split()) >= 2), lines[0] if lines else None)
    pairs = {}
    for l in lines:
        m = re.match(r"^([A-Za-z][A-Za-z ./&()'-]{1,40}?)\s*[:;]\s*(.+)$", l)
        if m:
            key, val = m.group(1).strip(), m.group(2).strip(" _.")
            if val and key.lower() not in pairs and len(val) < 120:
                pairs[key.lower()] = val
        if len(pairs) >= 10:
            break
    return {
        "baslik": title,
        "alanlar": pairs or None,
        "tarih": find_date(text),
    }


_CLOSING = re.compile(r"^\s*(?:sincerely|yours\s+(?:truly|sincerely|faithfully|very\s+truly)|very\s+truly\s+yours|"
                      r"respectfully(?:\s+yours)?|best\s+regards|kind\s+regards|regards|cordially)\b", re.I)
_SALUTATION = re.compile(r"^\s*dear\s+(.{2,60}?)\s*[:,;]?\s*$", re.I | re.M)


def extract_letter(text):
    """Mektup / resmi yazı (dilekçe): kurum (antet), tarih, alıcı, hitap, konu, imza sahibi."""
    lines = _lines(text)
    salute = _first(_SALUTATION, text, 1)
    sal_idx = next((i for i, l in enumerate(lines) if _SALUTATION.match(l)), None)
    # imza: kapanış ifadesinden sonraki ilk isim gibi görünen satır
    signer = None
    close_idx = next((i for i, l in enumerate(lines) if _CLOSING.match(l)), None)
    if close_idx is not None:
        signer = next((l for l in lines[close_idx + 1:close_idx + 6] if _looks_like_name(l)), None)
        if signer is None:
            persons = _ents("\n".join(lines[close_idx + 1:close_idx + 6]), {"PERSON"})
            signer = persons[0] if persons else None
    # alıcı adresi: tarih ile "Dear" arasındaki blokta isim gibi görünen ilk satır
    head = [l for l in lines[:4] if not DATE.search(l)]
    org = next((l for l in head if _CORP.search(l)), None)
    if org is None:
        orgs = _ents("\n".join(head), {"ORG"})
        org = orgs[0] if orgs else None
    recipient = None
    if sal_idx:
        date_idx = next((i for i, l in enumerate(lines[:sal_idx]) if DATE.search(l)), -1)
        window = lines[max(date_idx + 1, sal_idx - 8):sal_idx]
        recipient = next((l for l in window if _looks_like_name(l) and l != org), None)
    return {
        "kurum": org,
        "tarih": find_date(text),
        "alici": recipient,
        "hitap": salute,
        "konu": _header(text, "re|subject|subj"),
        "imza": signer,
    }


_DATELINE = re.compile(r"^([A-Z][A-Z .,'-]{2,40})\s*(?:\((?:AP|UPI|Reuters|AFP)\))?\s*[-—–]{1,2}\s", re.M)


def extract_news(text):
    """Gazete haberi: başlık, yer (dateline), tarih, haberde geçen kişi ve kurumlar."""
    lines = _lines(text)
    title = next((l for l in lines[:6] if 3 <= len(l.split()) <= 15 and sum(c.isalpha() for c in l) > 10), None)
    return {
        "baslik": title,
        "yer": _first(_DATELINE, text, 1),
        "tarih": find_date(text),
        "kisiler": _ents(text, {"PERSON"})[:4] or None,
        "kurumlar": _ents(text, {"ORG"})[:4] or None,
    }


_NOT_TITLE = re.compile(r"\b(?:printed\s+in|journal|vol\.?|volume|pp\.|fax|copyright|received|accepted|reprint(?:ed)?|"
                        r"sonderdruck|page|inc\.?|wiley|elsevier|press)\b|©|\(c\)", re.I)
_AFFIL = re.compile(r"\b(?:university|universit|college|institute|department|dept|school|hospital|laborator|"
                    r"centre|center|company|corporation|inc)\b", re.I)
_VOCAB = None


def _vocab():
    """İngilizce kelime listesi (SymSpell sıklık sözlüğü); yoksa boş küme → sözlük kontrolü atlanır."""
    global _VOCAB
    if _VOCAB is None:
        try:
            from importlib import resources
            path = resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt"
            _VOCAB = {l.split()[0] for l in path.read_text().splitlines() if l.strip()}
        except Exception:
            _VOCAB = set()
    return _VOCAB


def _wordlike_ratio(line):
    words = re.findall(r"[A-Za-z]{2,}", line)
    if not words:
        return 0.0
    vocab = _vocab()
    if not vocab:   # sözlük yoksa: sesli harf içeren, makul uzunlukta kelime oranı
        return sum(bool(re.search(r"[aeiouy]", w, re.I)) and len(w) <= 15 for w in words) / len(words)
    return sum(w.lower() in vocab for w in words) / len(words)


def _is_title_line(l):
    words = l.split()
    letters = sum(c.isalpha() for c in l)
    alpha_words = re.findall(r"[A-Za-z]+", l)
    mean_len = sum(map(len, alpha_words)) / max(1, len(alpha_words))   # OCR çöpü kısa parçalardan oluşur
    return (len(words) >= 3 and letters >= 15 and letters / max(1, len(l.replace(" ", ""))) >= 0.85
            and mean_len >= 4 and not _NOT_TITLE.search(l) and not _AFFIL.search(l) and _wordlike_ratio(l) >= 0.7)


def _looks_like_author_line(l):
    """"Pauline M. Tang, Judith Finkelstein" gibi: virgülle ayrılmış, büyük harfle başlayan ad grupları."""
    parts = [p.strip() for p in re.split(r",|\band\b", l) if p.strip()]
    return len(parts) >= 2 and sum(bool(re.match(r"^[A-Z][\w.'-]*(?:\s+[A-Z][\w.'-]*){1,3}$", p)) for p in parts) >= 2


def _scientific_title(lines):
    """v2: dergi künyesi, faks başlığı, "Printed in Great Britain" gibi satırları atlar; başlık iki satıra
    bölünmüşse birleştirir."""
    for i, l in enumerate(lines[:15]):
        if _is_title_line(l) and len(l.split()) >= 4:
            title, j = l, i + 1
            while j < min(i + 3, len(lines)) and _is_title_line(lines[j]) and not re.match(r"^\s*by\b", lines[j], re.I) \
                    and not _looks_like_author_line(lines[j]) and not title.rstrip().endswith((".", ":")):
                title += " " + lines[j]
                j += 1
            prev = lines[i - 1] if i > 0 else ""
            if prev and prev.isupper() and l.isupper() and 1 <= len(prev.split()) <= 6 and not _NOT_TITLE.search(prev) \
                    and sum(c.isalpha() for c in prev) >= 5 and _wordlike_ratio(prev) >= 0.7:
                title = prev + " " + title           # "CHARACTERIZATION" + "OF AIRBORNE PARTICULATE MATTER"
            return title.strip(), j
    return None, 0


def _scientific_authors(lines, after):
    """Başlıktan sonraki 3 satırda: "BY ..." satırı ya da virgül/"and" ile ayrılmış adlar (kurum satırı değil)."""
    for l in lines[after:after + 3]:
        if re.fullmatch(r"[\d\s.,()-]+", l) or _AFFIL.search(l):
            continue
        if re.search(r"[\"*]\s*$", l) or re.match(r"^\W*(?:[IVX]+\.|\d+\.|\.)\s", l):   # alt başlık: "I. Vapor Phase Analysis*"
            continue
        m = re.match(r"^\s*by\s+(.+)$", l, re.I)
        cand = m.group(1) if m else l
        cand = re.split(r",\s*(?:PH|M\.?\s?D|Ph\.?\s?D|F\.?A\.?C\.?P)\b", cand)[0]   # unvanları at
        if m or re.search(r",|\band\b", cand):
            parts = [p.strip(" .") for p in re.split(r",|\band\b", cand) if len(p.strip(" .")) > 1]
            if parts and sum(bool(re.search(r"[A-Za-z]{2,}", p)) for p in parts) >= max(1, len(parts) // 2):
                return parts[:5]
        caps = [w for w in cand.split() if w[:1].isupper() or re.fullmatch(r"\d\.", w)]   # OCR: "J." → "3."
        stop = {"in", "of", "the", "for", "on", "at", "to", "from", "with", "by"}
        if 2 <= len(cand.split()) <= 5 and len(caps) == len(cand.split()) \
                and not any(w.lower() in stop for w in cand.split()):   # tek yazar: "DAVID J. LEWIS"
            return [cand.strip()]
    return None


def extract_scientific(text):
    """Bilimsel makale: başlık, yazarlar, özet (Abstract), anahtar kelimeler."""
    lines = _lines(text)
    title, after = _scientific_title(lines)
    authors = _scientific_authors(lines, after) if title else None
    if title is None:   # eski yöntem
        title = next((l for l in lines[:6] if len(l.split()) >= 4 and sum(c.isalpha() for c in l) > 15), None)
    if authors is None and title is None:
        authors = [p for p in _ents("\n".join(lines[:12]), {"PERSON"})][:5] or None
    m = re.search(r"\babstract\b[\s:.-]*(.{20,600}?)(?:\n\s*\n|\b(?:introduction|key\s*words?|keywords)\b|$)",
                  text, re.I | re.S)
    abstract = " ".join(m.group(1).split())[:300] if m else None
    kw = _first(re.compile(r"\bkey\s*words?\s*[:.-]\s*(.+)", re.I), text, 1)
    return {
        "baslik": title,
        "yazarlar": authors,
        "ozet": abstract,
        "anahtar_kelimeler": kw,
        "kaynakca_var": bool(re.search(r"\b(?:references|bibliography|literature cited)\b", text, re.I)),
    }


EXTRACTORS = {"resume": extract_resume, "invoice": extract_invoice, "email": extract_email, "form": extract_form,
              "letter": extract_letter, "news_article": extract_news, "scientific_publication": extract_scientific}
TR_NAMES = {"resume": "CV", "invoice": "fatura", "email": "e-posta", "form": "form",
            "letter": "resmi yazı / mektup", "news_article": "gazete haberi", "scientific_publication": "bilimsel makale"}


def summarize(label, fields):
    f = fields
    if label == "resume":
        parts = [f.get("ad_soyad"), f.get("universite"),
                 f.get("bolum_derece") if f.get("bolum_derece") != f.get("universite") else None]
        if f.get("beceriler"):
            parts.append("Beceriler: " + ", ".join(f["beceriler"][:5]))
    elif label == "invoice":
        parts = [f"Fatura no {f['fatura_no']}" if f.get("fatura_no") else None, f.get("firma"), f.get("tarih"),
                 f"Toplam {f['toplam_tutar']}" if f.get("toplam_tutar") else None]
    elif label == "email":
        who = " → ".join(x for x in (f.get("gonderen"), f.get("alici")) if x) or None
        parts = [who, f"Konu: {f['konu']}" if f.get("konu") else None, f.get("tarih")]
    elif label == "letter":
        parts = [f.get("kurum"), f.get("tarih"), f"Alıcı: {f.get('hitap') or f.get('alici')}" if (f.get("hitap") or f.get("alici")) else None,
                 f"Konu: {f['konu']}" if f.get("konu") else None, f"İmza: {f['imza']}" if f.get("imza") else None]
    elif label == "news_article":
        parts = [f.get("baslik"), f.get("yer"), f.get("tarih"),
                 "Kişiler: " + ", ".join(f["kisiler"][:3]) if f.get("kisiler") else None]
    elif label == "scientific_publication":
        parts = [f.get("baslik"), "Yazarlar: " + ", ".join(f["yazarlar"][:3]) if f.get("yazarlar") else None,
                 f"Anahtar kelimeler: {f['anahtar_kelimeler']}" if f.get("anahtar_kelimeler") else None]
    else:
        kv = list((f.get("alanlar") or {}).items())[:4]
        parts = [f.get("baslik")] + [f"{k} = {v}" for k, v in kv]
    parts = [p for p in parts if p]
    return f"Bu bir {TR_NAMES.get(label, label)}" + (": " + " · ".join(parts) if parts else " (içerik alanları okunamadı)")


def extract(text, label):
    fields = EXTRACTORS.get(label, extract_form)(text or "")
    return {"alanlar": fields, "ozet": summarize(label, fields)}


# ---------------------------------------------------------------- konu özeti (danışmanın "ikinci aşama" hedefi)
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

_GENERIC = {"new", "please", "thank", "thanks", "dear", "regards", "sincerely", "page", "shows", "also",
            "would", "could", "like", "just", "know", "let", "use", "used", "make", "need", "want", "good",
            "subject", "cc", "bcc", "sent", "fax", "tel", "phone", "date", "re", "fwd", "attached"}
_STOP = sorted(ENGLISH_STOP_WORDS | _GENERIC)

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")


class TopicSummarizer:
    """Belgenin ne hakkında olduğunu anlatan anahtar kelimeler + 2-3 cümlelik çıkarımsal özet.

    Kelime ağırlıkları (TF-IDF) eğitim belgelerinden öğrenilir: bir belgede sık geçen ama
    tüm belgelerde yaygın olmayan kelimeler o belgenin konusunu temsil eder. OCR çöpü genelde
    tek bir belgede geçtiği için min_df=2 ile sözlüğe girmez.
    """

    def __init__(self, min_df=2, max_df=0.5):
        self.vec = TfidfVectorizer(stop_words=_STOP, ngram_range=(1, 2), min_df=min_df, max_df=max_df,
                                   sublinear_tf=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b")

    def fit(self, texts):
        self.vec.fit([t for t in texts if isinstance(t, str)])
        self.vocab_ = self.vec.vocabulary_
        self.names_ = self.vec.get_feature_names_out()
        return self

    def keywords(self, text, k=6):
        if not text:
            return []
        row = self.vec.transform([text]).toarray().ravel()
        order = np.argsort(row)[::-1]
        out = []
        for i in order:
            if row[i] <= 0 or len(out) >= k:
                break
            term = self.names_[i]
            if any(term in o or o in term for o in out):   # "invoice" ile "invoice number" tekrarını önle
                continue
            out.append(term)
        return out

    def _sentences(self, text):
        sents = []
        for s in _SENT_SPLIT.split(text):
            s = " ".join(s.split())
            words = s.split()
            alpha = sum(w.isalpha() for w in words)
            if len(words) >= 5 and alpha / len(words) >= 0.6 and len(s) <= 300:
                sents.append(s)
        return sents

    def summary(self, text, n=2):
        sents = self._sentences(text or "")
        if not sents:
            return []
        m = self.vec.transform(sents)
        scores = np.asarray(m.sum(axis=1)).ravel() / np.sqrt(np.maximum(m.getnnz(axis=1), 1))
        top = sorted(np.argsort(scores)[::-1][:n])
        return [sents[i] for i in top if scores[i] > 0]

    def describe(self, text, label, k=6, n=2):
        kws = self.keywords(text, k)
        sents = self.summary(text, n)
        tur = TR_NAMES.get(label, label)
        line = f"Bu bir {tur}."
        if kws:
            line += f" Konusu / anahtar kelimeler: {', '.join(kws)}."
        return {"anahtar_kelimeler": kws, "ozet_cumleler": sents, "konu_ozeti": line}


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_core.py
import cv2
import numpy as np
import pytesseract


def estimate_skew(gray, max_angle=5.0, step=0.25):
    """Projection-profile skew estimate: the angle whose row sums are sharpest."""
    small = cv2.resize(gray, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    th = cv2.medianBlur(th, 3)  # drop isolated noise pixels
    h, w = th.shape
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(-max_angle, max_angle + 1e-9, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        r = cv2.warpAffine(th, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(r.sum(axis=1, dtype=np.float64)))
        if score > best_score:
            best_angle, best_score = float(a), score
    return best_angle


def deskew(gray):
    """Rotate the page so text lines are horizontal."""
    angle = estimate_skew(gray)
    if abs(angle) < 0.3:
        return gray, 0.0
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def preprocess(img_bgr_or_gray, upscale_to=2000, do_deskew=True, binarize=True):
    """Grayscale -> upscale (RVL-CDIP is max 1000px) -> denoise -> deskew -> Otsu."""
    img = img_bgr_or_gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    gray = cv2.medianBlur(gray, 3)  # salt-and-pepper scan noise, before upscaling
    h, w = gray.shape
    scale = upscale_to / max(h, w)
    if scale > 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    angle = 0.0
    if do_deskew:
        gray, angle = deskew(gray)
    if binarize:
        _, gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray, angle


def run_ocr(img, lang="eng", psm=3, min_conf=0):
    """Tesseract OCR -> words, normalized boxes (0-1000, LayoutLM format), line text."""
    h, w = img.shape[:2]
    d = pytesseract.image_to_data(img, lang=lang, config=f"--oem 1 --psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    words, boxes, confs, lines = [], [], [], {}
    for i, txt in enumerate(d["text"]):
        txt = txt.strip()
        conf = float(d["conf"][i])
        if not txt or conf < min_conf:
            continue
        x, y, bw, bh = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        words.append(txt)
        confs.append(conf)
        boxes.append([int(1000 * x / w), int(1000 * y / h),
                      int(1000 * (x + bw) / w), int(1000 * (y + bh) / h)])
        key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(txt)
    text = "\n".join(" ".join(v) for _, v in sorted(lines.items()))
    return {
        "text": text,
        "words": words,
        "boxes": boxes,
        "confs": confs,
        "n_words": len(words),
        "mean_conf": float(np.mean(confs)) if confs else 0.0,
    }


def detect_table(gray):
    """Morfolojik işlemlerle yatay/dikey çizgileri bulup belgede tablo olup olmadığını tahmin eder.

    OCR tabloları bozarak okuduğu için (danışman/asistan görüşmesi) tablo bilgisi
    görüntüden ayrıca çıkarılır ve sınıflandırmada ek özellik olarak kullanılabilir.
    """
    img = gray if gray.ndim == 2 else cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    h, w = img.shape
    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(10, w // 25), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(10, h // 25)))
    horiz = cv2.morphologyEx(th, cv2.MORPH_OPEN, hk)
    vert = cv2.morphologyEx(th, cv2.MORPH_OPEN, vk)
    n_h = len(cv2.findContours(horiz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    n_v = len(cv2.findContours(vert, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    inter = cv2.bitwise_and(cv2.dilate(horiz, np.ones((3, 3), np.uint8)), cv2.dilate(vert, np.ones((3, 3), np.uint8)))
    n_x = len(cv2.findContours(inter, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    line_ratio = float((horiz > 0).mean() + (vert > 0).mean())
    return {
        "n_hlines": n_h,
        "n_vlines": n_v,
        "n_intersections": n_x,
        "line_ratio": round(line_ratio, 5),
        "has_table": bool(n_h >= 3 and n_v >= 2 and n_x >= 4),
    }


def rotate_bound(gray, angle):
    """Görüntüyü kırpmadan döndürür (köşeler beyazla doldurulur)."""
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    return cv2.warpAffine(gray, M, (nw, nh), flags=cv2.INTER_CUBIC, borderValue=255)


def _good_words(o, min_conf=70):
    return sum(c >= min_conf for c in o["confs"])


def upscale_if_small(gray, ocr, target_px=24):
    """Yazı küçükse görüntüyü büyütüp yeniden okur (21 Ekim).

    Tesseract harf yüksekliği ~20-30 piksel olduğunda en iyi çalışır. RVL-CDIP taramalarında kelime yüksekliği
    7-12 piksel; internetten alınan küçük resimlerde daha da az. Kelime kutularının ortanca yüksekliği
    hedefin altındaysa görüntü büyütülür; güvenli (>= 70) okunan kelime sayısı artarsa büyütülmüş okuma seçilir.
    """
    hs = [b[3] - b[1] for b in ocr["boxes"]]
    if not hs:
        med = 0.0
    else:
        med = float(np.median(hs)) * gray.shape[0] / 1000.0
    if med >= target_px * 0.8 and _good_words(ocr) > 0:
        return gray, ocr, 1.0
    f = 2.0   # denemelerde x2 en dengelisi: x3 bazı tablo sütunlarını kaybettirdi
    big = cv2.resize(gray, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    o2 = run_ocr(big)
    return (big, o2, f) if _good_words(o2) > 1.1 * _good_words(ocr) else (gray, ocr, 1.0)


def ocr_with_orientation(img, upscale=True):
    """Yan, eğik ya da ters dönmüş sayfayı düzeltip OCR yapar.

    21 Ekim: elle değerlendirmede ~77° dönük bir fatura hiç okunamamıştı ("WoRsN"). Yazı satırlarının açısı,
    satır toplamlarının en keskin olduğu açı aranarak (-90°…+90°) bulunur; satır yönü 180° belirsiz olduğu için
    sayfa o açıyla ve ters yönüyle okunur. Açı küçükse Tesseract'ın yön tespiti (OSD) baş aşağı sayfaya bakar.
    Aday hâller arasından en çok ve en emin kelimeyi veren seçilir; asıl hâlden en az %20 iyi değilse sayfaya
    dokunulmaz, böylece düz belgeler bozulmaz.
    Dönüş: (görüntü, ocr sonucu, döndürme açısı)
    """
    import re
    gray = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ocr = run_ocr(gray)
    score = lambda o: o["n_words"] * o["mean_conf"]
    angles = []
    a = estimate_skew(gray, max_angle=90, step=1.0)
    if abs(a) >= 3:
        angles += [a, a + 180]
    else:
        try:
            osd = pytesseract.image_to_osd(gray, config="--psm 0 -c min_characters_to_try=5")
            if int(re.search(r"Rotate: (\d+)", osd).group(1)) == 180:
                angles.append(180)
        except Exception:
            pass
    best = (gray, ocr, 0)
    for ang in angles:
        r = rotate_bound(gray, ang)
        o = run_ocr(r)
        if score(o) > 1.2 * score(best[1]):
            best = (r, o, ang)
    if upscale:
        g2, o2, _ = upscale_if_small(best[0], best[1])
        best = (g2, o2, best[2])
    return best


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/sablon.py
"""Belge türü şablonları (danışman isteği, 15 Ekim): her türün hemen her belgesinde bulunan ortak ifadeler.

OCR metninde bir türün şablon ifadeleri bulunuyorsa bu, makine öğrenmesi modelinin kararını destekleyen
ikinci bir kanıttır (ör. faturada "invoice", "total", para tutarı; mektupta "Dear", "Sincerely").
Veri seti İngilizce olduğu için ifadeler İngilizcedir; Türkçe karşılıkları yorumda verilmiştir.
"""
import re

# her ifade: (kısa ad, düzenli ifade) · metin küçük harfe çevrilmiş temiz OCR metnine uygulanır
TEMPLATES = {
    "email": [  # e-posta başlık alanları
        ("from:", r"^\s*from\s*[:;]"),
        ("to:", r"^\s*to\s*[:;]"),
        ("subject:", r"^\s*(?:subject|subj)\s*[:;]"),
        ("sent/date:", r"^\s*(?:sent|date)\s*[:;]"),
        ("cc:", r"^\s*b?cc\s*[:;]"),
        ("@ adresi", r"[\w.+-]+@[\w-]+"),
        ("message", r"\bmessage\b"),
    ],
    "invoice": [  # fatura: numara, tutar, miktar, ödeme
        ("invoice", r"\binvoice\b"),
        ("total", r"\btotal\b"),
        ("amount", r"\bamount\b"),
        ("para tutarı", r"[$€£]\s?\d|\b\d{1,3}(?:,\d{3})*\.\d{2}\b"),
        ("qty/quantity", r"\b(?:qty|quantity)\b"),
        ("price", r"\bprice\b"),
        ("bill/ship/sold to", r"\b(?:bill|ship|sold|remit)\s*to\b"),
        ("payment/due", r"\b(?:payment|payable|due|terms)\b"),
    ],
    "resume": [  # CV bölüm başlıkları
        ("education", r"\beducation\b"),
        ("experience", r"\b(?:experience|employment)\b"),
        ("university/college", r"\b(?:university|college|institute)\b"),
        ("degree", r"\b(?:b\.?s\.?|m\.?s\.?|ph\.?d\.?|bachelor|master|doctor|degree)\b"),
        ("born/birth", r"\b(?:born|birth|married)\b"),
        ("research", r"\bresearch\b"),
        ("publications/references", r"\b(?:publications|references|honors|awards)\b"),
        # 21 Ekim: CV'yi bilimsel makaleden ayıran, CV'ye özgü ifadeler eklendi (university korunuyor)
        ("biographical sketch/cv", r"\b(?:biographical\s+sketch|curriculum\s+vitae|resume)\b"),
        ("position title", r"\bposition\s+title\b"),
        ("professor", r"\b(?:assistant|associate|full|visiting)?\s*professor\b"),
        ("yıl aralığı", r"\b(?:19|20)\d{2}\s*[-–]\s*(?:(?:19|20)?\d{2}\b|present)"),
        ("present", r"\bpresent\b"),
    ],
    "form": [  # form: faks kapağı, doldurulacak alanlar
        ("fax/facsimile", r"\b(?:fax|facsimile|telecopy|telecopier)\b"),
        ("transmission", r"\btransmi(?:ssion|tted|t)\b"),
        ("pages", r"\bpages?\b"),
        ("name:", r"\bname\s*[:;]"),
        ("date:", r"\bdate\s*[:;]"),
        ("signature/approved", r"\b(?:signature|signed|approv(?:ed|al))\b"),
        ("request/number", r"\b(?:request|number|no\.)\b"),
    ],
    "letter": [  # mektup / resmi yazı (dilekçe): hitap, kapanış, ek
        ("dear", r"^\s*dear\b"),
        ("sincerely/regards", r"\b(?:sincerely|yours\s+truly|very\s+truly|regards|respectfully|cordially)\b"),
        ("re:", r"^\s*re\s*[:;]"),
        ("enclosed/enclosure", r"\benclos(?:ed|ure)\b"),
        ("thank you", r"\bthank\s+you\b"),
        ("letter", r"\bletter\b"),
        ("please", r"\bplease\b"),
    ],
    "news_article": [  # gazete haberi
        ("said", r"\bsaid\b"),
        ("ajans (AP/UPI/Reuters)", r"\((?:ap|upi)\)|\breuters\b|\bassociated press\b"),
        ("percent", r"\bpercent\b"),
        ("according to", r"\baccording to\b"),
        ("spokesman/officials", r"\b(?:spokesman|spokeswoman|officials?)\b"),
        ("yesterday/today", r"\b(?:yesterday|today|last week)\b"),
        ("times/post/journal", r"\b(?:times|post|journal|tribune|herald|news)\b"),
    ],
    "scientific_publication": [  # bilimsel makale bölümleri
        ("abstract", r"\babstract\b"),
        ("introduction", r"\bintroduction\b"),
        ("methods/results", r"\b(?:methods?|results)\b"),
        ("references", r"\breferences\b"),
        ("et al.", r"\bet\s+al\b"),
        ("fig./table", r"\b(?:fig|figure|table)\b"),
        ("journal/vol.", r"\b(?:journal|vol|volume)\b"),
    ],
}

# İlk sürüm (20 Ekim): karşılaştırma için saklanır. CV şablonunda sadece ilk 7 ifade vardı.
_NEW_RESUME = {"biographical sketch/cv", "position title", "professor", "yıl aralığı", "present"}
TEMPLATES_V1 = {c: [(n, p) for n, p in pats if not (c == "resume" and n in _NEW_RESUME)]
                for c, pats in TEMPLATES.items()}

_CACHE = {}


def _compiled(templates):
    key = id(templates)
    if key not in _CACHE:
        _CACHE[key] = {c: [(n, re.compile(p, re.I | re.M)) for n, p in pats] for c, pats in templates.items()}
    return _CACHE[key]


_COMPILED = _compiled(TEMPLATES)


def match_template(text, classes=None, templates=None):
    """{sınıf: [eşleşen ifade adları]}"""
    comp = _compiled(templates or TEMPLATES)
    text = text or ""
    return {c: [n for n, rx in comp[c] if rx.search(text)] for c in (classes or comp)}


def fit_weights(texts, labels, classes, templates=None):
    """Her ifadeye veriden ağırlık: (kendi sınıfında geçtiği belge oranı) − (diğer sınıflardaki en yüksek oran).

    Örnek: "education" CV'lerin %59'unda, makalelerin %1'inde geçer → ağırlık ≈ 0,58.
    "university" CV'lerin %93'ünde, makalelerin %36'sında geçer → ağırlık ≈ 0,57'den düşük kalır ve
    tek başına makaleyi CV'ye çeviremez. Negatif çıkanlar küçük bir pozitif değere (0,01) çekilir.
    """
    templates = templates or TEMPLATES
    labels = list(labels)
    matches = [match_template(t, list(templates), templates) for t in texts]
    rate = {}
    for c in templates:
        for n, _ in templates[c]:
            rate[(c, n)] = {k: (sum(1 for m, l in zip(matches, labels) if l == k and n in m[c]) /
                                max(1, sum(1 for l in labels if l == k))) for k in classes}
    weights = {}
    for c in classes:
        weights[c] = {}
        for n, _ in templates[c]:
            r = rate[(c, n)]
            others = max((r[k] for k in classes if k != c), default=0.0)
            weights[c][n] = round(max(0.01, r[c] - others), 4)
    return weights


def template_scores(text, classes=None, weights=None, templates=None):
    """{sınıf: şablon puanı (0-1)}. weights verilmezse her ifade eşit sayılır (eşleşen ifade oranı)."""
    templates = templates or TEMPLATES
    m = match_template(text, classes, templates)
    if weights is None:
        return {c: len(v) / len(templates[c]) for c, v in m.items()}
    return {c: sum(weights[c].get(n, 0) for n in v) / max(1e-9, sum(weights[c].values())) for c, v in m.items()}


def template_vector(text, classes):
    """Model için özellik vektörü: her ifade için 0/1 + her sınıf için eşleşme oranı."""
    m = match_template(text, classes)
    bits = [float(n in m[c]) for c in classes for n, _ in _COMPILED[c]]
    return bits + [len(m[c]) / len(TEMPLATES[c]) for c in classes]


def template_feature_names(classes):
    return [f"{c}:{n}" for c in classes for n, _ in _COMPILED[c]] + [f"{c}:oran" for c in classes]


def rule_predict(text, classes, weights=None, templates=None):
    """Sadece şablona göre tahmin: en yüksek puanlı sınıf (hiç eşleşme yoksa None)."""
    s = template_scores(text, classes, weights, templates)
    best = max(s, key=s.get)
    return best if s[best] > 0 else None


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/final_model.py
"""Son model: karakter TF-IDF + tablo özellikleri → olasılık veren Linear SVM (Platt kalibrasyonu).

04'teki ek deneyde tablo özelliği (asistan önerisi) Linear SVM'in test başarısını artırdı. Bu modül o
deneyi demoda kullanılabilecek tek bir modele dönüştürür:

    metin (normalize_for_model çıktısı) ──► karakter TF-IDF ─┐
                                                              ├─► Linear SVM (kalibre) ─► olasılıklar
    görüntü ──► detect_table ──► 5 tablo özelliği (ölçekli) ─┘

"Emin değilim": en yüksek olasılık `threshold`'un altındaysa model karar vermez; en olası iki türü
gösterir. Eşik validation setinde seçilir (test setine bakılmaz).
"""
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

TAB_COLS = ["n_hlines", "n_vlines", "n_intersections", "line_ratio", "has_table"]
UNSURE = "emin_degilim"


def char_tfidf():
    # 04'teki karakter TF-IDF ile aynı ayarlar
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.9,
                           sublinear_tf=True, max_features=100_000)


def tab_array(rows):
    """detect_table sözlükleri ya da TAB_COLS sütunlu DataFrame → (n, 5) dizi; sayılar log ölçekte."""
    if hasattr(rows, "loc"):
        t = rows[TAB_COLS].astype(float).values.copy()
    else:
        t = np.array([[float(r[c]) for c in TAB_COLS] for r in rows])
    t[:, :3] = np.log1p(t[:, :3])
    return t


class TextTableModel:
    def __init__(self, C=0.5, table_weight=0.2, threshold=0.0, seed=42):
        self.C, self.table_weight, self.threshold, self.seed = C, table_weight, threshold, seed

    def _X(self, texts, tabs, fit=False):
        if fit:
            self.vec_ = char_tfidf().fit(texts)
            self.scaler_ = StandardScaler().fit(tab_array(tabs))
        X_text = self.vec_.transform(texts)
        if self.table_weight == 0:
            return X_text.tocsr()
        X_tab = csr_matrix(self.table_weight * self.scaler_.transform(tab_array(tabs)))
        return hstack([X_text, X_tab]).tocsr()

    def fit(self, texts, tabs, y, calibrate=True):
        X = self._X(texts, tabs, fit=True)
        svm = LinearSVC(C=self.C, random_state=self.seed)
        self.clf_ = CalibratedClassifierCV(svm, cv=5, method="sigmoid").fit(X, y) if calibrate else svm.fit(X, y)
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, texts, tabs):
        return self.clf_.predict_proba(self._X(texts, tabs))

    def predict(self, texts, tabs):
        return self.clf_.predict(self._X(texts, tabs))

    def decide(self, texts, tabs, threshold=None):
        """Tahmin + 'emin değilim': en yüksek olasılık eşiğin altındaysa UNSURE döner."""
        t = self.threshold if threshold is None else threshold
        P = self.predict_proba(texts, tabs)
        top = P.argmax(axis=1)
        labels = self.classes_[top].astype(object)
        labels[P.max(axis=1) < t] = UNSURE
        return labels, P


def coverage_table(y_true, P, classes, thresholds):
    """Her eşik için: kapsama (karar verilen belge oranı), karar verilenlerdeki doğruluk, kaçan hata sayısı."""
    y_true = np.asarray(y_true)
    pred = np.asarray(classes)[P.argmax(axis=1)]
    conf = P.max(axis=1)
    rows = []
    for t in thresholds:
        keep = conf >= t
        n_err_all = int((pred != y_true).sum())
        n_err_kept = int((pred[keep] != y_true[keep]).sum())
        rows.append({
            "esik": t,
            "kapsama": keep.mean(),
            "karar_verilenlerde_dogruluk": (pred[keep] == y_true[keep]).mean() if keep.any() else np.nan,
            "emin_degilim_sayisi": int((~keep).sum()),
            "yakalanan_hata": n_err_all - n_err_kept,
            "kalan_hata": n_err_kept,
            "gereksiz_kararsizlik": int(((~keep) & (pred == y_true)).sum()),
        })
    return rows


In [ ]:
import ocr_core, text_clean, extract, sablon, final_model
for m in (ocr_core, text_clean, extract, sablon, final_model): importlib.reload(m)
from sablon import match_template, template_scores
from ocr_core import run_ocr, detect_table
from text_clean import clean_text, normalize_for_model
from extract import extract as extract_fields, TR_NAMES, TopicSummarizer
print("spaCy yüklü:", extract._NLP is not None)

## 3. Model: olasılık veren sürüm

04'teki en iyi model (Linear SVM) olasılık vermez, sadece karar skoru verir. Demoda "%95 emin" diyebilmek için aynı model
**olasılık kalibrasyonu** (Platt yöntemi, 5 katlı) ile train verisinde yeniden eğitilir. Tahminler aynı kalır; test başarısı aşağıda kontrol edilir.

In [ ]:
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

base = joblib.load(MODELS / "best_model.joblib")
info = json.loads((MODELS / "best_model.json").read_text())
# Veri: cleaned_dataset.csv (03) yoksa ocr_dataset.csv'den (02) aynı temizlik burada yapılır
def load_ocr_data():
    """02'nin OCR çıktısı + 06'da OCR'lanan aday sınıflar → sadece CLASSES'taki sınıflar."""
    parts = [pd.read_csv(DATA / "ocr_dataset.csv")]
    if (DATA / "ocr_dataset_aday.csv").exists():
        parts.append(pd.read_csv(DATA / "ocr_dataset_aday.csv"))
    d = pd.concat(parts, ignore_index=True).drop_duplicates("path")
    d = d[d["label"].isin(CLASSES)].reset_index(drop=True)
    d["label_id"] = d["label"].map(CLASSES.index)
    d["text"] = d["text"].fillna("")
    missing = set(CLASSES) - set(d["label"])
    assert not missing, f"OCR verisi olmayan sınıf: {missing} → önce 06_sinif_secimi'ni çalıştırın."
    return d

df = pd.read_csv(DATA / "cleaned_dataset.csv") if (DATA / "cleaned_dataset.csv").exists() else None
if df is not None and set(CLASSES) <= set(df["label"]):
    df = df[df["label"].isin(CLASSES)].reset_index(drop=True)
elif (DATA / "ocr_dataset.csv").exists():
    print("cleaned_dataset.csv yok ya da sınıflar değişmiş → OCR çıktısından temizleniyor")
    df = load_ocr_data()
    df["text_clean"] = df["text"].fillna("").map(clean_text)
    df["text_norm"] = df["text_clean"].map(normalize_for_model)
    df["is_empty"] = df["text_clean"].str.split().str.len().fillna(0) < 5
else:
    df = None
    print("UYARI: Ne cleaned_dataset.csv ne ocr_dataset.csv bulundu. Demo 04 modeliyle çalışır; 5. ve 7. bölümler atlanır.")

summarizer = None
if df is not None:
    df["text_norm"] = df["text_norm"].fillna("")
    train = df[(df.split == "train") & (~df.is_empty)]
    test = df[df.split == "test"]
    model = CalibratedClassifierCV(clone(base), cv=5, method="sigmoid").fit(train.text_norm, train.label)
    for name, m in [("04 modeli", base), ("kalibre model", model)]:
        p = m.predict(test.text_norm)
        print(f"{name:14s} test accuracy {accuracy_score(test.label, p):.4f} · macro-F1 {f1_score(test.label, p, average='macro'):.4f}")
    summarizer = TopicSummarizer().fit(train["text_clean"].fillna(""))   # konu özeti için kelime ağırlıkları
    try:
        joblib.dump(model, MODELS / "demo_model.joblib")
    except OSError as e:
        print("Model Drive'a kaydedilemedi (Drive dolu olabilir):", e)
elif (MODELS / "demo_model.joblib").exists():
    model = joblib.load(MODELS / "demo_model.joblib")
else:
    from scipy.special import softmax
    class ScoreModel:            # SVM karar skorlarını 0-1 aralığına çevirir (olasılık değil, göreli skor)
        def __init__(self, m): self.m, self.classes_ = m, m.classes_
        def predict(self, X): return self.m.predict(X)
        def predict_proba(self, X): return softmax(self.m.decision_function(X), axis=1)
    model = base if hasattr(base, "predict_proba") else ScoreModel(base)
    test = None
print("Model:", info["model"], "·", info["özellik"], "TF-IDF")

# 10_son_model çalıştırıldıysa: metin + tablo özellikli son model ve "emin değilim" eşiği kullanılır
FINAL, FINAL_INFO = None, {}
if (MODELS / "final_model.joblib").exists():
    FINAL = joblib.load(MODELS / "final_model.joblib")
    FINAL_INFO = json.loads((MODELS / "final_model.json").read_text())
    if set(FINAL.classes_) != set(CLASSES):
        print("UYARI: son model farklı sınıflarla eğitilmiş → 10'u yeniden çalıştırın. Şimdilik metin modeli kullanılıyor.")
        FINAL = None
if FINAL is not None:
    print(f"Son model kullanılıyor: {FINAL_INFO['özellik']} · test accuracy %{100 * FINAL_INFO['test_accuracy']:.2f} · "
          f"emin değilim eşiği {FINAL_INFO['emin_degilim_esigi']}")
else:
    print("10_son_model bulunamadı → metin modeli kullanılıyor (tablo özelliği ve 'emin değilim' yok).")

## 4. Uçtan uca fonksiyon

In [ ]:
import re
import pymupdf   # PDF'i resme çevirir (sistem programı gerektirmez)

def load_document(path):
    """Resim veya PDF (ilk sayfa) → tek kanallı görüntü (renkli yazı koyulaştırılır)."""
    path = str(path)
    if path.lower().endswith(".pdf"):
        with pymupdf.open(path) as doc:
            pix = doc[0].get_pixmap(dpi=200, colorspace=pymupdf.csRGB)
        arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.stride)
        img = arr[:, :pix.width * 3].reshape(pix.height, pix.width, 3).copy()
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise ValueError(f"Okunamadı: {path}")
    # Renkli yazı (mavi/yeşil tutarlar) gri tonlamada soluk kalıyor; her pikselin en koyu kanalı alınınca
    # yazı siyaha yakın çıkıyor (21 Ekim, internetten alınan faturada 80.000,00 → "$0.000.00" okunuyordu).
    # Gri tonlamalı taramalarda (RVL-CDIP) sonuç değişmez.
    return img.min(axis=2)

from ocr_core import ocr_with_orientation   # dönük sayfa düzeltme (21 Ekim)

def analyze(img, min_conf=0.5):
    """Görüntü → OCR (gerekirse yön düzeltme) → temizleme → tür tahmini → içerik çıkarımı."""
    img, ocr, rotated = ocr_with_orientation(img)        # yön düzeltme + yazı küçükse x2 büyütme (21 Ekim)
    clean = clean_text(ocr["text"])
    if len(clean.split()) < 5:
        return {"tur": None, "olasiliklar": {}, "ozet": "Belgeden yeterli metin okunamadı.",
                "alanlar": {}, "konu": {}, "sablon": "", "tablo": detect_table(img), "ocr_metni": ocr["text"], "ocr_guveni": ocr["mean_conf"]}
    table = detect_table(img)
    if FINAL is not None:                                # 10: metin + tablo, kalibre olasılık
        proba = FINAL.predict_proba([normalize_for_model(clean)], [table])[0]
        probs = dict(zip(FINAL.classes_, proba.round(4)))
        min_conf = FINAL_INFO.get("emin_degilim_esigi", min_conf)
    else:
        proba = model.predict_proba([normalize_for_model(clean)])[0]
        probs = dict(zip(model.classes_, proba.round(4)))
    label = max(probs, key=probs.get)
    unsure = probs[label] < min_conf
    ext = extract_fields(clean, label)
    note = ""
    if unsure:                                           # "emin değilim": en olası iki tür
        (l1, p1), (l2, p2) = sorted(probs.items(), key=lambda x: -x[1])[:2]
        note = (f"Emin değilim — büyük ihtimalle {TR_NAMES.get(l1, l1)} (%{100 * p1:.0f}) "
                f"ya da {TR_NAMES.get(l2, l2)} (%{100 * p2:.0f}). ")
    if rotated:
        note += f"(Sayfa {rotated:.0f}° döndürülerek okundu.) "
    topic = summarizer.describe(clean, label) if summarizer is not None else {}
    # Şablon desteği (danışman isteği 4): tahmin edilen türün ortak ifadeleri metinde var mı?
    matches = match_template(clean.lower(), list(model.classes_))
    t_scores = template_scores(clean.lower(), list(model.classes_))
    t_best = max(t_scores, key=t_scores.get)
    if matches[label]:
        t_note = f"Şablon destekliyor: {', '.join(matches[label])}"
    elif t_scores[t_best] > 0:
        t_note = f"Şablon farklı bir tür gösteriyor: {TR_NAMES.get(t_best, t_best)} ({', '.join(matches[t_best])})"
    else:
        t_note = "Şablon ifadesi bulunamadı"
    return {"tur": label, "emin_degil": bool(unsure), "olasiliklar": probs, "ozet": note + ext["ozet"],
            "alanlar": ext["alanlar"], "konu": topic, "tablo": table, "sablon": t_note,
            "ocr_metni": clean, "ocr_guveni": round(ocr["mean_conf"], 1)}

def show(path, res, true_label=None):
    fig, ax = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={"width_ratios": [1, 1.2]})
    ax[0].imshow(load_document(path), cmap="gray"); ax[0].axis("off")
    title = f"Tahmin: {res['tur']}" + (f"  ·  Gerçek: {true_label}" if true_label else "")
    ax[0].set_title(title, color="green" if (true_label is None or true_label == res["tur"]) else "red", fontsize=13)
    ax[1].axis("off")
    probs = "  ".join(f"{k}: %{100 * v:.0f}" for k, v in sorted(res["olasiliklar"].items(), key=lambda x: -x[1]))
    fields = "\n".join(f"{k}: {v}" for k, v in res["alanlar"].items())
    topic = res.get("konu") or {}
    sents = "\n".join("• " + x[:110] for x in topic.get("ozet_cumleler", []))
    tbl = "var" if res.get("tablo", {}).get("has_table") else "yok"
    txt = (f"{res['ozet']}\n\n{topic.get('konu_ozeti', '')}\n{sents}\n\nTablo: {tbl}\n{res.get('sablon', '')}\n"
           f"Olasılıklar: {probs}\nOCR güveni: {res['ocr_guveni']}\n\nAlanlar:\n{fields}")
    ax[1].text(0, 1, txt.replace("$", r"\$"), va="top", fontsize=10, wrap=True, family="monospace")   # "$" matematik kipini açmasın
    plt.tight_layout(); plt.show()

## 5. RVL-CDIP test belgeleri üzerinde

Modelin eğitimde **görmediği** test belgelerinden her türden ikişer örnek.

In [ ]:
if test is None or not (DATA / "images").exists():
    print("Test verisi/görüntüleri bulunamadı, bu bölüm atlandı.")
    samples = pd.DataFrame()
else:
    samples = pd.concat([g[~g.is_empty].sample(2, random_state=SEED) for _, g in test.groupby("label")])
for _, r in samples.iterrows():
    res = analyze(load_document(DATA / r.path))
    print(f"[{r.label}] {Path(r.path).name}\n  {res['ozet']}\n")
    show(DATA / r.path, res, true_label=r.label)

## 6. Kendi belgeleriniz

1. Drive'da `bitirme_ocr/kendi_belgelerim/` klasörüne **İngilizce** belgeler koyun (JPG, PNG veya PDF): kendi CV'niz, bir fatura, yazdırılmış bir e-posta, bir form
2. Hücreyi çalıştırın

Kişisel bilgi içeren belgeleri rapora/sunuma koymadan önce maskeleyin.

In [ ]:
files = sorted(p for p in OWN.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".pdf"})
if not files:
    print(f"{OWN} klasörü boş. Belge ekleyip tekrar çalıştırın.")
own_rows = []
for p in files:
    res = analyze(load_document(p))
    own_rows.append({"dosya": p.name, "tahmin": res["tur"], "olasılık": max(res["olasiliklar"].values(), default=0),
                     "özet": res["ozet"]})
    show(p, res)
if own_rows:
    display(pd.DataFrame(own_rows))

## 7. İçerik çıkarımının değerlendirilmesi

Çıkarılan alanların doğruluğu, **türü doğru tahmin edilen** test belgelerinden her türden 5 tane (toplam 20) üzerinde elle değerlendirilir.

Her alan için belgeye bakıp seçin:

| Model bir değer çıkardıysa | Model boş bıraktıysa |
|---|---|
| **doğru** · **yanlış** | **belgede de yok** · **kaçırıldı** |

| Ölçüt | Formül |
|---|---|
| Precision | doğru / (doğru + yanlış) |
| Recall | doğru / (doğru + kaçırıldı) |

Kayıtlar her belgede `cikarim_degerlendirme.csv` dosyasına yazılır; yarıda bırakıp sonra devam edebilirsiniz.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

EVAL_FILE = ROOT / "cikarim_degerlendirme.csv"
if test is None:
    raise SystemExit("Test verisi yok, değerlendirme atlandı.")
pred = model.predict(test.text_norm)
pool = test[(pred == test.label.values) & (~test.is_empty.values)]
eval_docs = pd.concat([g.sample(min(5, len(g)), random_state=SEED) for _, g in pool.groupby("label")])
done = pd.read_csv(EVAL_FILE) if EVAL_FILE.exists() else pd.DataFrame(columns=["belge", "tur", "alan", "deger", "karar"])
todo = [r for _, r in eval_docs.iterrows() if r.path not in set(done.belge)]
print(f"Değerlendirilecek: {len(todo)} belge (tamamlanan: {done.belge.nunique()})")

def fmt(v):
    return ", ".join(map(str, v)) if isinstance(v, list) else (json.dumps(v, ensure_ascii=False) if isinstance(v, dict) else str(v))

out = w.Output()
state = {"i": 0, "widgets": {}, "res": None}

def render():
    with out:
        clear_output(wait=True)
        if state["i"] >= len(todo):
            print("Bitti! Sonuçlar için bir sonraki hücreyi çalıştırın."); return
        r = todo[state["i"]]
        res = analyze(load_document(DATA / r.path)); state["res"] = res
        fig, ax = plt.subplots(figsize=(7, 9)); ax.imshow(load_document(DATA / r.path), cmap="gray"); ax.axis("off")
        ax.set_title(f"{state['i'] + 1}/{len(todo)} · {r.label} · {Path(r.path).name}"); plt.show()
        state["widgets"] = {}
        rows = []
        for k, v in res["alanlar"].items():
            empty = v in (None, [], {})
            opts = ["belgede de yok", "kaçırıldı"] if empty else ["doğru", "yanlış"]
            tb = w.ToggleButtons(options=opts, value=opts[0])
            state["widgets"][k] = (tb, None if empty else fmt(v))
            rows.append(w.HBox([w.HTML(f"<b style='display:inline-block;width:130px'>{k}</b>"),
                                w.HTML(f"<span style='display:inline-block;width:360px'>{'—' if empty else fmt(v)[:80]}</span>"), tb]))
        display(w.VBox(rows))

def save_next(_):
    global done
    if state["i"] >= len(todo): return
    r = todo[state["i"]]
    new = [{"belge": r.path, "tur": r.label, "alan": k, "deger": val, "karar": tb.value}
           for k, (tb, val) in state["widgets"].items()]
    done = pd.concat([done, pd.DataFrame(new)], ignore_index=True)
    done.to_csv(EVAL_FILE, index=False)
    state["i"] += 1; render()

btn = w.Button(description="Kaydet ve sonraki →", button_style="success")
btn.on_click(save_next)
display(btn, out); render()

In [ ]:
ev = pd.read_csv(EVAL_FILE) if EVAL_FILE.exists() else pd.DataFrame()
if ev.empty:
    print("Henüz değerlendirme yok.")
else:
    def scores(g):
        c = g.karar.value_counts()
        d, y, k = c.get("doğru", 0), c.get("yanlış", 0), c.get("kaçırıldı", 0)
        return pd.Series({"doğru": d, "yanlış": y, "kaçırıldı": k,
                          "precision": d / (d + y) if d + y else np.nan, "recall": d / (d + k) if d + k else np.nan})
    by_field = ev.groupby(["tur", "alan"]).apply(scores).round(3)
    by_type = ev.groupby("tur").apply(scores).round(3)
    display(by_type); display(by_field)
    by_field.to_csv(ROOT / "cikarim_sonuclari.csv")
    print(f"Değerlendirilen belge: {ev.belge.nunique()}")

## 8. Demo arayüzü (Gradio)

Hücre çalışınca bir **public link** (`https://....gradio.live`) çıkar. Bu linki tarayıcıda açıp belge yükleyebilir, hocanıza da gönderebilirsiniz.
Link Colab oturumu açık kaldığı sürece (en fazla ~72 saat) çalışır.

In [ ]:
import gradio as gr

import traceback

def demo_fn(file):
    if file is None:
        return {}, "Bir belge yükleyin.", {}, ""
    try:
        path = file if isinstance(file, str) else getattr(file, "name", file)
        res = analyze(load_document(path))
        probs = {TR_NAMES.get(k, k): float(v) for k, v in res["olasiliklar"].items()}
        fields = json.loads(json.dumps(res["alanlar"], ensure_ascii=False, default=str))
        topic = res.get("konu") or {}
        sents = "\n".join(f"- {x}" for x in topic.get("ozet_cumleler", []))
        tbl = "Belgede tablo **var**." if res.get("tablo", {}).get("has_table") else "Belgede tablo yok."
        md_out = f"### {res['ozet']}\n\n**{topic.get('konu_ozeti', '')}**\n\n{sents}\n\n{tbl}\n\n{res.get('sablon', '')}"
        return probs, md_out, fields, res["ocr_metni"]
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)                                   # Colab hücre çıktısına da yazılır
        return {}, f"**Hata:** `{type(e).__name__}: {e}`\n\n```\n{tb[-1500:]}\n```", {}, ""

# Önce Gradio olmadan bir test: kendi_belgelerim'deki ilk dosya
_first = next((p for p in sorted(OWN.iterdir()) if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".pdf"}), None)
if _first:
    print("Test:", _first.name, "→", demo_fn(str(_first))[1][:300])

app = gr.Interface(
    fn=demo_fn,
    inputs=gr.File(label="Belge (JPG, PNG veya PDF)", file_types=["image", ".pdf"]),
    outputs=[gr.Label(num_top_classes=4, label="Belge türü"), gr.Markdown(label="Özet"),
             gr.JSON(label="Çıkarılan alanlar"), gr.Textbox(label="OCR metni", lines=10)],
    title="OCR Tabanlı Belge Sınıflandırma ve İçerik Çıkarımı",
    description=" · ".join(TR_NAMES.get(c, c) for c in CLASSES) + " — İngilizce belgeler için. OCR: Tesseract · Sınıflandırma: "
                + (f"karakter TF-IDF + tablo özelliği, {FINAL_INFO['model']}" if FINAL is not None else "TF-IDF + " + info["model"]),
    flagging_mode="never",
)
app.launch(share=True, debug=False, show_error=True)

## Çıktılar

| Dosya | İçerik |
|---|---|
| `models/demo_model.joblib` | Olasılık veren model |
| `extract.py` | İçerik çıkarım kuralları |
| `cikarim_degerlendirme.csv`, `cikarim_sonuclari.csv` | Elle değerlendirme ve alan bazında precision / recall |